In [ ]:
# Cell 1 — Installs
!pip install transformers datasets evaluate jiwer peft torch torchaudio -q

In [ ]:
# Cell 2 — Imports
import torch
import numpy as np
import copy
import time
import gc
from datasets import load_dataset, Dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cuda


In [ ]:
# Cell 3 — Load dataset
dataset = load_dataset("DTU54DL/common-accent", "default")
df = dataset["train"].to_pandas()
print("Dataset loaded!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Dataset loaded!


In [ ]:
# Cell 4 — Split accents
df_a = df[df["accent"] == "German English,Non native speaker"].reset_index(drop=True)
df_b = df[df["accent"] == "India and South Asia (India, Pakistan, Sri Lanka)"].reset_index(drop=True)

accent_a_ds = Dataset.from_pandas(df_a).cast_column("audio", Audio(sampling_rate=16000))
accent_b_ds = Dataset.from_pandas(df_b).cast_column("audio", Audio(sampling_rate=16000))

N_TRAIN, N_EVAL = 500, 150

accent_a_train = accent_a_ds.select(range(N_TRAIN))
accent_a_eval  = accent_a_ds.select(range(N_TRAIN, N_TRAIN + N_EVAL))
accent_b_train = accent_b_ds.select(range(N_TRAIN))
accent_b_eval  = accent_b_ds.select(range(N_TRAIN, N_TRAIN + N_EVAL))

print(f"Train A: {len(accent_a_train)}, Eval A: {len(accent_a_eval)}")
print(f"Train B: {len(accent_b_train)}, Eval B: {len(accent_b_eval)}")

Train A: 500, Eval A: 150
Train B: 500, Eval B: 150


In [ ]:
# Cell 5 — Load Whisper
MODEL_NAME = "openai/whisper-small"
processor  = WhisperProcessor.from_pretrained(MODEL_NAME)
print("Whisper loaded!")

preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Whisper loaded!


In [ ]:
# Cell 6 — Preprocessing
def preprocess(batch):
    audio = batch["audio"]["array"]
    sr    = batch["audio"]["sampling_rate"]
    batch["input_features"] = processor.feature_extractor(
        audio, sampling_rate=sr, return_tensors="pt"
    ).input_features[0]
    batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids
    return batch

accent_a_train = accent_a_train.map(preprocess, remove_columns=accent_a_train.column_names)
accent_a_eval  = accent_a_eval.map(preprocess,  remove_columns=accent_a_eval.column_names)
accent_b_train = accent_b_train.map(preprocess, remove_columns=accent_b_train.column_names)
accent_b_eval  = accent_b_eval.map(preprocess,  remove_columns=accent_b_eval.column_names)
print("Preprocessing done!")

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

Preprocessing done!


In [ ]:
# Cell 7 — Collator
@dataclass
class SpeechDataCollator:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch   = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

data_collator = SpeechDataCollator(
    processor=processor,
    decoder_start_token_id=processor.tokenizer.convert_tokens_to_ids("<|startoftranscript|>")
)
print("Collator ready!")

Collator ready!


In [ ]:
# Cell 8 — WER + Training args
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {"wer": round(wer_metric.compute(predictions=pred_str, references=label_str), 4)}

def get_training_args(output_dir, max_steps=100):
    return Seq2SeqTrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=1e-4,
        warmup_steps=20,
        max_steps=max_steps,
        fp16=torch.cuda.is_available(),
        eval_strategy="steps",
        eval_steps=100,
        save_steps=100,
        logging_steps=50,
        predict_with_generate=True,
        generation_max_length=128,
        report_to="none",
        load_best_model_at_end=True,
        metric_for_best_model="wer",
        greater_is_better=False,
    )

print("WER + training args ready!")

WER + training args ready!


In [ ]:
# Cell 9 — Restore baseline results (already ran, don't waste time rerunning)
all_results = {}
all_results["Baseline"] = {
    "wer_A_after_phase1": 0.3006,
    "wer_A_after_phase2": 0.3154,
    "wer_B_after_phase2": 0.3559,
    "forgetting":         0.0148,
    "model_size_mb":      966.9,
    "train_time":         0,
}
print("Baseline results restored! Now run EWC directly.")# Cell 8 — Data collator
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class SpeechDataCollator:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch   = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = SpeechDataCollator(
    processor=processor,
    decoder_start_token_id=processor.tokenizer.convert_tokens_to_ids("<|startoftranscript|>")
)
print("Data collator ready!")

Baseline results restored! Now run EWC directly.
Data collator ready!


In [ ]:
# Cell 9 — WER metric
import evaluate
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str  = processor.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": round(wer, 4)}

print("WER metric ready!")

WER metric ready!


In [ ]:
#BASELINE (fixed generation config)
import time
from transformers import Seq2SeqTrainer

all_results = {}

print("=" * 60)
print("METHOD 1: BASELINE (Naive Fine-Tuning)")
print("=" * 60)

model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)
# Fix — set on generation_config not model.config
model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens    = []

# Phase 1: Train on Accent A
trainer_a = Seq2SeqTrainer(
    model=model,
    args=get_training_args("./baseline_phase1", max_steps=100),
    train_dataset=accent_a_train,
    eval_dataset=accent_a_eval,
    processing_class=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
print("\n[Phase 1] Training on Accent A (German English)...")
trainer_a.train()

eval_a1 = trainer_a.evaluate(accent_a_eval)
wer_a1  = eval_a1["eval_wer"]
print(f"WER on A after phase 1: {wer_a1:.4f}")

# Phase 2: Fine-tune on Accent B
print("\n[Phase 2] Fine-tuning on Accent B (Indian English)...")
t0 = time.time()
trainer_b = Seq2SeqTrainer(
    model=model,
    args=get_training_args("./baseline_phase2", max_steps=100),
    train_dataset=accent_b_train,
    eval_dataset=accent_b_eval,
    processing_class=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer_b.train()
train_time = time.time() - t0

eval_a2 = trainer_b.evaluate(accent_a_eval)
eval_b2 = trainer_b.evaluate(accent_b_eval)

all_results["Baseline"] = {
    "wer_A_after_phase1": wer_a1,
    "wer_A_after_phase2": eval_a2["eval_wer"],
    "wer_B_after_phase2": eval_b2["eval_wer"],
    "forgetting":         eval_a2["eval_wer"] - wer_a1,
    "model_size_mb":      sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6,
    "train_time":         train_time,
}

print(f"\n── Baseline Results ──")
print(f"WER on A after training B: {all_results['Baseline']['wer_A_after_phase2']:.4f}  (was {wer_a1:.4f})")
print(f"WER on B:                  {all_results['Baseline']['wer_B_after_phase2']:.4f}")
print(f"Forgetting:                {all_results['Baseline']['forgetting']:+.4f}")
print(f"Model size:                {all_results['Baseline']['model_size_mb']:.1f} MB")

METHOD 1: BASELINE (Naive Fine-Tuning)


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]


[Phase 1] Training on Accent A (German English)...


Step,Training Loss,Validation Loss,Wer
100,1.191472,0.746505,0.300600


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


WER on A after phase 1: 0.3006

[Phase 2] Fine-tuning on Accent B (Indian English)...


Step,Training Loss,Validation Loss,Wer
100,1.227033,0.964101,0.355900


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].



── Baseline Results ──
WER on A after training B: 0.3154  (was 0.3006)
WER on B:                  0.3559
Forgetting:                +0.0148
Model size:                966.9 MB


In [ ]:
# Cell 10 — EWC class
class EWC:
    def __init__(self, model, dataset, data_collator, n_samples=200):
        self.model  = model
        self.params = {n: p.clone().detach() for n, p in model.named_parameters() if p.requires_grad}
        self.fisher = self._compute_fisher(dataset, data_collator, n_samples)

    def _compute_fisher(self, dataset, data_collator, n_samples):
        print("Computing Fisher Information...")
        fisher = {n: torch.zeros_like(p) for n, p in self.model.named_parameters() if p.requires_grad}
        loader = torch.utils.data.DataLoader(
            dataset.select(range(min(n_samples, len(dataset)))),
            batch_size=2,
            collate_fn=data_collator
        )
        self.model.train()
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            self.model.zero_grad()
            outputs = self.model(**batch)
            outputs.loss.backward()
            for n, p in self.model.named_parameters():
                if p.requires_grad and p.grad is not None:
                    fisher[n] += p.grad.detach() ** 2
        for n in fisher:
            fisher[n] /= len(loader)
        print("Fisher Information computed!")
        return fisher

    def penalty(self, model, ewc_lambda=500):
        loss = 0.0
        for n, p in model.named_parameters():
            if n in self.fisher:
                loss += (self.fisher[n] * (p - self.params[n]) ** 2).sum()
        return ewc_lambda * loss

print("EWC class ready!")

EWC class ready!


In [ ]:
# Cell 11 — EWC Trainer
class EWCTrainer(Seq2SeqTrainer):
    def __init__(self, ewc_instance, ewc_lambda=500, **kwargs):
        super().__init__(**kwargs)
        self.ewc        = ewc_instance
        self.ewc_lambda = ewc_lambda

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs    = model(**inputs)
        task_loss  = outputs.loss
        ewc_loss   = self.ewc.penalty(model, self.ewc_lambda)
        total_loss = task_loss + ewc_loss
        return (total_loss, outputs) if return_outputs else total_loss

print("EWC Trainer ready!")

EWC Trainer ready!


In [ ]:
# Cell 12 — Run EWC
print("=" * 60)
print("METHOD 2: EWC")
print("=" * 60)

model_ewc = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)
model_ewc.generation_config.forced_decoder_ids = None
model_ewc.generation_config.suppress_tokens    = []

# Phase 1: Train on A
trainer_ewc_a = Seq2SeqTrainer(
    model=model_ewc,
    args=get_training_args("./ewc_phase1", max_steps=100),
    train_dataset=accent_a_train,
    eval_dataset=accent_a_eval,
    processing_class=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
print("\n[Phase 1] Training on Accent A...")
trainer_ewc_a.train()

eval_a1    = trainer_ewc_a.evaluate(accent_a_eval)
wer_a1_ewc = eval_a1["eval_wer"]
print(f"WER on A after phase 1: {wer_a1_ewc:.4f}")

# Free trainer from GPU before Fisher computation
del trainer_ewc_a
gc.collect()
torch.cuda.empty_cache()

# Compute Fisher on A
ewc = EWC(model_ewc, accent_a_train, data_collator, n_samples=150)

# Phase 2: Train on B with EWC penalty (lambda=500)
print("\n[Phase 2] Training on Accent B with EWC constraint...")
t0 = time.time()
trainer_ewc_b = EWCTrainer(
    ewc_instance=ewc,
    ewc_lambda=500,
    model=model_ewc,
    args=get_training_args("./ewc_phase2", max_steps=100),
    train_dataset=accent_b_train,
    eval_dataset=accent_b_eval,
    processing_class=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer_ewc_b.train()
train_time = time.time() - t0

eval_a2 = trainer_ewc_b.evaluate(accent_a_eval)
eval_b2 = trainer_ewc_b.evaluate(accent_b_eval)

all_results["EWC"] = {
    "wer_A_after_phase1": wer_a1_ewc,
    "wer_A_after_phase2": eval_a2["eval_wer"],
    "wer_B_after_phase2": eval_b2["eval_wer"],
    "forgetting":         eval_a2["eval_wer"] - wer_a1_ewc,
    "model_size_mb":      966.9,
    "train_time":         train_time,
}

# Free memory immediately
del model_ewc, trainer_ewc_b
gc.collect()
torch.cuda.empty_cache()

print(f"\n── EWC Results ──")
print(f"WER on A after training B: {all_results['EWC']['wer_A_after_phase2']:.4f}  (was {wer_a1_ewc:.4f})")
print(f"WER on B:                  {all_results['EWC']['wer_B_after_phase2']:.4f}")
print(f"Forgetting:                {all_results['EWC']['forgetting']:+.4f}  ← should be less than +0.0148")
print(f"Model size:                {all_results['EWC']['model_size_mb']:.1f} MB")

METHOD 2: EWC


model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]


[Phase 1] Training on Accent A...


Step,Training Loss,Validation Loss,Wer
100,2.549196,0.745198,0.325000


... (4 outputs left out)

Step,Training Loss,Validation Loss,Wer
100,2.495384,0.960756,0.404600


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].



── EWC Results ──
WER on A after training B: 0.3539  (was 0.3250)
WER on B:                  0.4046
Forgetting:                +0.0289  ← should be less than +0.0148
Model size:                966.9 MB


In [ ]:
# Cell 13 — Run LoRA
from peft import LoraConfig, get_peft_model, TaskType

print("=" * 60)
print("METHOD 3: LoRA")
print("=" * 60)

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)

base = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
base.generation_config.forced_decoder_ids = None
base.generation_config.suppress_tokens    = []
model_lora = get_peft_model(base, lora_config).to(device)

trainable = sum(p.numel() for p in model_lora.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model_lora.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

# Phase 1: Train on A
trainer_lora_a = Seq2SeqTrainer(
    model=model_lora,
    args=get_training_args("./lora_phase1", max_steps=100),
    train_dataset=accent_a_train,
    eval_dataset=accent_a_eval,
    processing_class=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
print("\n[Phase 1] Training LoRA adapters on Accent A...")
trainer_lora_a.train()

eval_a1      = trainer_lora_a.evaluate(accent_a_eval)
wer_a1_lora  = eval_a1["eval_wer"]
print(f"WER on A after phase 1: {wer_a1_lora:.4f}")

# Save adapter A weights
model_lora.save_pretrained("./lora_adapter_a")

del trainer_lora_a
gc.collect()
torch.cuda.empty_cache()

# Phase 2: Fresh LoRA adapters on B (base stays frozen)
print("\n[Phase 2] Training fresh LoRA adapters on Accent B...")
base2 = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
base2.generation_config.forced_decoder_ids = None
base2.generation_config.suppress_tokens    = []
model_lora_b = get_peft_model(base2, lora_config).to(device)

t0 = time.time()
trainer_lora_b = Seq2SeqTrainer(
    model=model_lora_b,
    args=get_training_args("./lora_phase2", max_steps=100),
    train_dataset=accent_b_train,
    eval_dataset=accent_b_eval,
    processing_class=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer_lora_b.train()
train_time = time.time() - t0

# Evaluate B with accent B adapters
eval_b2 = trainer_lora_b.evaluate(accent_b_eval)

# Evaluate A by reloading accent A adapters
del model_lora_b, trainer_lora_b
gc.collect()
torch.cuda.empty_cache()

from peft import PeftModel
base3 = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
base3.generation_config.forced_decoder_ids = None
base3.generation_config.suppress_tokens    = []
model_lora_a = PeftModel.from_pretrained(base3, "./lora_adapter_a").to(device)

eval_a2 = Seq2SeqTrainer(
    model=model_lora_a,
    args=get_training_args("./lora_eval", max_steps=1),
    eval_dataset=accent_a_eval,
    processing_class=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
).evaluate(accent_a_eval)

all_results["LoRA"] = {
    "wer_A_after_phase1": wer_a1_lora,
    "wer_A_after_phase2": eval_a2["eval_wer"],
    "wer_B_after_phase2": eval_b2["eval_wer"],
    "forgetting":         eval_a2["eval_wer"] - wer_a1_lora,
    "model_size_mb":      trainable * 4 / 1e6,   # only adapter weights
    "train_time":         train_time,
}

del model_lora_a
gc.collect()
torch.cuda.empty_cache()

print(f"\n── LoRA Results ──")
print(f"WER on A after training B: {all_results['LoRA']['wer_A_after_phase2']:.4f}  (was {wer_a1_lora:.4f})")
print(f"WER on B:                  {all_results['LoRA']['wer_B_after_phase2']:.4f}")
print(f"Forgetting:                {all_results['LoRA']['forgetting']:+.4f}  ← should be close to 0")
print(f"Adapter size:              {all_results['LoRA']['model_size_mb']:.2f} MB  ← much smaller than 966 MB!")

METHOD 3: LoRA


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Trainable params: 884,736 / 242,619,648 (0.36%)

[Phase 1] Training LoRA adapters on Accent A...


Step,Training Loss,Validation Loss,Wer
100,3.225086,0.448352,0.116900


WER on A after phase 1: 0.1169

[Phase 2] Training fresh LoRA adapters on Accent B...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss,Wer
100,3.236642,0.486501,0.161900


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]


── LoRA Results ──
WER on A after training B: 0.1169  (was 0.1169)
WER on B:                  0.1619
Forgetting:                +0.0000  ← should be close to 0
Adapter size:              3.54 MB  ← much smaller than 966 MB!


In [ ]:
# Cell 14 — Run Hybrid (LoRA + EWC)
print("=" * 60)
print("METHOD 4: HYBRID (LoRA + EWC)")
print("=" * 60)

class HybridEWCTrainer(Seq2SeqTrainer):
    def __init__(self, ewc_instance, ewc_lambda=500, **kwargs):
        super().__init__(**kwargs)
        self.ewc        = ewc_instance
        self.ewc_lambda = ewc_lambda

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs    = model(**inputs)
        task_loss  = outputs.loss
        ewc_loss   = self.ewc.penalty(model, self.ewc_lambda)
        total_loss = task_loss + ewc_loss
        return (total_loss, outputs) if return_outputs else total_loss

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)

base_h = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
base_h.generation_config.forced_decoder_ids = None
base_h.generation_config.suppress_tokens    = []
model_hybrid = get_peft_model(base_h, lora_config).to(device)

trainable_h = sum(p.numel() for p in model_hybrid.parameters() if p.requires_grad)

# Phase 1: Train on A
trainer_hybrid_a = Seq2SeqTrainer(
    model=model_hybrid,
    args=get_training_args("./hybrid_phase1", max_steps=100),
    train_dataset=accent_a_train,
    eval_dataset=accent_a_eval,
    processing_class=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
print("\n[Phase 1] Training LoRA adapters on Accent A...")
trainer_hybrid_a.train()

eval_a1        = trainer_hybrid_a.evaluate(accent_a_eval)
wer_a1_hybrid  = eval_a1["eval_wer"]
print(f"WER on A after phase 1: {wer_a1_hybrid:.4f}")

del trainer_hybrid_a
gc.collect()
torch.cuda.empty_cache()

# Compute Fisher on adapter weights
ewc_hybrid = EWC(model_hybrid, accent_a_train, data_collator, n_samples=150)

# Phase 2: Train on B with EWC on adapter weights
print("\n[Phase 2] Training on Accent B with LoRA + EWC...")
t0 = time.time()
trainer_hybrid_b = HybridEWCTrainer(
    ewc_instance=ewc_hybrid,
    ewc_lambda=500,
    model=model_hybrid,
    args=get_training_args("./hybrid_phase2", max_steps=100),
    train_dataset=accent_b_train,
    eval_dataset=accent_b_eval,
    processing_class=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer_hybrid_b.train()
train_time = time.time() - t0

eval_a2 = trainer_hybrid_b.evaluate(accent_a_eval)
eval_b2 = trainer_hybrid_b.evaluate(accent_b_eval)

all_results["Hybrid"] = {
    "wer_A_after_phase1": wer_a1_hybrid,
    "wer_A_after_phase2": eval_a2["eval_wer"],
    "wer_B_after_phase2": eval_b2["eval_wer"],
    "forgetting":         eval_a2["eval_wer"] - wer_a1_hybrid,
    "model_size_mb":      trainable_h * 4 / 1e6,
    "train_time":         train_time,
}

del model_hybrid, trainer_hybrid_b
gc.collect()
torch.cuda.empty_cache()

print(f"\n── Hybrid Results ──")
print(f"WER on A after training B: {all_results['Hybrid']['wer_A_after_phase2']:.4f}  (was {wer_a1_hybrid:.4f})")
print(f"WER on B:                  {all_results['Hybrid']['wer_B_after_phase2']:.4f}")
print(f"Forgetting:                {all_results['Hybrid']['forgetting']:+.4f}")
print(f"Adapter size:              {all_results['Hybrid']['model_size_mb']:.2f} MB")

METHOD 4: HYBRID (LoRA + EWC)


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]


[Phase 1] Training LoRA adapters on Accent A...


Step,Training Loss,Validation Loss,Wer
100,3.016138,0.400490,0.118200


WER on A after phase 1: 0.1182
Computing Fisher Information...
Fisher Information computed!

[Phase 2] Training on Accent B with LoRA + EWC...


Step,Training Loss,Validation Loss,Wer
100,1.866689,0.397521,0.160100



── Hybrid Results ──
WER on A after training B: 0.1092  (was 0.1182)
WER on B:                  0.1601
Forgetting:                -0.0090
Adapter size:              3.54 MB


In [ ]:
# Cell 15 — Final comparison table
print("\n" + "=" * 85)
print(f"{'METHOD':<12} {'WER-A↓':>8} {'WER-B↓':>8} {'FORGETTING↓':>13} {'SIZE(MB)↓':>11} {'TIME(s)':>9}")
print("=" * 85)

for name, res in all_results.items():
    print(
        f"{name:<12}"
        f"{res['wer_A_after_phase2']:>8.4f}"
        f"{res['wer_B_after_phase2']:>8.4f}"
        f"{res['forgetting']:>13.4f}"
        f"{res['model_size_mb']:>11.2f}"
        f"{res.get('train_time', 0):>9.1f}"
    )

print("=" * 85)
print("\nKey findings:")
best_forgetting = min(all_results.items(), key=lambda x: x[1]['forgetting'])
best_wer_a      = min(all_results.items(), key=lambda x: x[1]['wer_A_after_phase2'])
best_wer_b      = min(all_results.items(), key=lambda x: x[1]['wer_B_after_phase2'])
smallest_model  = min(all_results.items(), key=lambda x: x[1]['model_size_mb'])

print(f"  Best forgetting:  {best_forgetting[0]} ({best_forgetting[1]['forgetting']:+.4f})")
print(f"  Best WER on A:    {best_wer_a[0]} ({best_wer_a[1]['wer_A_after_phase2']:.4f})")
print(f"  Best WER on B:    {best_wer_b[0]} ({best_wer_b[1]['wer_B_after_phase2']:.4f})")
print(f"  Smallest model:   {smallest_model[0]} ({smallest_model[1]['model_size_mb']:.2f} MB)")


METHOD         WER-A↓   WER-B↓   FORGETTING↓   SIZE(MB)↓   TIME(s)
Baseline      0.3154  0.3559       0.0148     966.90      0.0
EWC           0.3539  0.4046       0.0289     966.90    381.4
LoRA          0.1169  0.1619       0.0000       3.54    260.7
Hybrid        0.1092  0.1601      -0.0090       3.54    272.9

Key findings:
  Best forgetting:  Hybrid (-0.0090)
  Best WER on A:    Hybrid (0.1092)
  Best WER on B:    Hybrid (0.1601)
  Smallest model:   LoRA (3.54 MB)


In [ ]:
# Cell 16 — Save results to JSON for your paper
import json

with open("results.json", "w") as f:
    json.dump(all_results, f, indent=2)
print("Saved to results.json!")

# Also print a latex-ready table for your paper
print("\n--- LaTeX table (copy into your paper) ---")
print("\\begin{table}[h]")
print("\\centering")
print("\\begin{tabular}{lcccc}")
print("\\hline")
print("Method & WER-A & WER-B & Forgetting & Size (MB) \\\\")
print("\\hline")
for name, res in all_results.items():
    print(f"{name} & {res['wer_A_after_phase2']:.4f} & {res['wer_B_after_phase2']:.4f} & {res['forgetting']:+.4f} & {res['model_size_mb']:.2f} \\\\")
print("\\hline")
print("\\end{tabular}")
print("\\caption{Comparison of continual learning methods for accent adaptation}")
print("\\label{tab:results}")
print("\\end{table}")

Saved to results.json!

--- LaTeX table (copy into your paper) ---
\begin{table}[h]
\centering
\begin{tabular}{lcccc}
\hline
Method & WER-A & WER-B & Forgetting & Size (MB) \\
\hline
Baseline & 0.3154 & 0.3559 & +0.0148 & 966.90 \\
EWC & 0.3539 & 0.4046 & +0.0289 & 966.90 \\
LoRA & 0.1169 & 0.1619 & +0.0000 & 3.54 \\
Hybrid & 0.1092 & 0.1601 & -0.0090 & 3.54 \\
\hline
\end{tabular}
\caption{Comparison of continual learning methods for accent adaptation}
\label{tab:results}
\end{table}
